# Proteome exploration with ESMC embeddings

Interactive UMAP over the per-protein mean-pooled ESMC embeddings, colored by
metadata pulled from Baserow.

**Prerequisites:** run the embedding step first so embeddings exist in the local
cache (or Baserow):

```bash
och-annotate embed -c ../config/octopus_chierchiae.yaml --dry-run   # preview
och-annotate embed -c ../config/octopus_chierchiae.yaml             # real run
```

Requires `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the environment (or a local `.env`).

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings, run_umap, plot_umap

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # set prefer_cache=False to pull from Baserow
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

In [ ]:
# UMAP across the whole proteome. cosine metric suits language-model embeddings.
coords = run_umap(df, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0)
coords[["umap_0", "umap_1"]].describe()

In [ ]:
# Color by any metadata column: chromosome, strand, gene_id, Ochierchiae_name, ...
fig = plot_umap(coords, color="chromosome", title=f"{cfg.name} proteome UMAP")
fig.show()

In [ ]:
# Optional: persist an interactive standalone HTML (the only disk write).
# from och_annotate.analysis import save_html
# save_html(fig, "octopus_chierchiae_umap.html")

## Leiden clustering + SAE-feature enrichment

Cluster proteins on the **ESMC embedding** kNN graph (UMAP is only for drawing),
then find which **SAE features** are enriched in each cluster — the scRNA-seq
marker-gene workflow with proteins as "cells" and SAE features as "genes".

Needs the clustering extras: `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph).

In [ ]:
import scanpy as sc
from och_annotate.analysis import build_anndata, sae_enrichment

# Only proteins that already have SAE features contribute to enrichment.
has_sae = df["sae_top_features"].notna()
print(f"{has_sae.sum()} / {len(df)} proteins have SAE features")
adata = build_anndata(df[has_sae].reset_index(drop=True))
print(adata)

# Cluster on the ESMC embedding kNN graph (NOT the 2-D UMAP); UMAP is just for layout.
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=15, metric="cosine")
sc.tl.leiden(adata, resolution=1.0, flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata)
adata.obs["leiden"].value_counts()

In [ ]:
# Visualize the clusters on the UMAP (computed above on the embedding graph).
sc.pl.umap(adata, color="leiden", legend_loc="on data", title="Leiden clusters")
# For context, also color by metadata, e.g.:
# sc.pl.umap(adata, color="chromosome")

In [ ]:
# Per-cluster enriched SAE features: Wilcoxon rank-sum on the activation matrix.
# (proteins = "cells", SAE features = "genes" -> standard marker-gene test.)
enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)
# columns: leiden, sae_feature, scores, logfoldchanges, pvals, pvals_adj
enrich.head(30)

In [ ]:
# Compact view: top-5 enriched SAE features per cluster
top5 = (enrich.sort_values(["leiden", "scores"], ascending=[True, False])
              .groupby("leiden", observed=True).head(5))
for cl, g in top5.groupby("leiden", observed=True):
    feats = ", ".join(f"{r.sae_feature}({r.scores:.1f})" for r in g.itertuples())
    print(f"cluster {cl}: {feats}")

# Dotplot of the marker SAE features across clusters
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

### Optional next steps

- **Read clusters biologically:** map the enriched feature ids to their
  natural-language descriptions from the EvolutionaryScale ESMC SAE Atlas, so each
  cluster reads as e.g. *"enriched for signal-peptide / zinc-finger features"*.
- **Persist clusters:** add a `leiden_cluster` long-text column in Baserow and write
  `adata.obs["leiden"]` back keyed by `transcript_id`
  (`BaserowClient.update_rows`). *(Creating the column needs a user/JWT token —
  the database token can edit rows but not add fields.)*
- **Tune granularity:** raise `resolution` for more clusters; adjust `n_neighbors`.
- **Sensitivity:** enrichment only sees each protein's stored **top-K** features —
  re-embed with a larger `--top-k` if you need finer feature coverage.